<a href="https://colab.research.google.com/github/disCrybee/Machine-Learning/blob/main/%D0%9B%D0%A0_8_%D0%9C%D0%9D_%D0%9A%D0%BB%D0%B0%D1%81%D1%82%D0%B5%D1%80%D0%B8%D0%B7%D0%B0%D1%86%D1%96%D1%8F_%D0%9F%D0%B5%D1%80%D0%B5%D1%85%D1%80%D0%B5%D1%81%D1%82%D0%90.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Лабораторна робота № 8
## Методи кластеризації: K-Means, метод ліктя, коефіцієнт силуета та PCA

**Мета:** навчитися застосовувати алгоритм K-Means, добирати кількість кластерів за допомогою методу ліктя та коефіцієнта силуета; дослідити використання PCA. Додатково — порівняти моделі класифікації на наборі даних про студентів.

**Середовище:** Google Colab або Jupyter Notebook із Python.

**Виконання:** запускайте комірки послідовно, зверху вниз.

## Завдання 1. Кластеризація набору даних Iris

**1.1. Імпорт бібліотек.**

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import numpy as np
from sklearn.datasets import load_iris
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

**1.2. Завантаження Iris.**

In [ ]:
iris = load_iris()
df_iris = pd.DataFrame(iris.data, columns=iris.feature_names)
df_iris['target'] = iris.target
print(df_iris.head())
print('\nРозмір:', df_iris.shape)
print('\nНазви ознак:', list(iris.feature_names))

**1.3. Масштабування та метод ліктя.** Використовуємо перші три ознаки (sepal length, sepal width, petal length).

In [ ]:
# Беремо три ознаки
X = df_iris.iloc[:, :3].values

# Стандартизація
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Метод ліктя (WCSS)
wcss = []
K_range = range(1, 11)
for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X_scaled)
    wcss.append(kmeans.inertia_)

plt.figure(figsize=(8, 5))
plt.plot(K_range, wcss, 'bo-')
plt.xlabel('Кількість кластерів (k)')
plt.ylabel('WCSS (Inertia)')
plt.title('Метод ліктя для Iris (3 ознаки)')
plt.xticks(K_range)
plt.grid(True, alpha=0.3)
plt.show()

print('WCSS для різних k:', [round(w, 2) for w in wcss])

**1.4. Метод силуетів.**

In [ ]:
sil_scores = []
K_sil = range(2, 11)
for k in K_sil:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans.fit_predict(X_scaled)
    score = silhouette_score(X_scaled, labels)
    sil_scores.append(score)

plt.figure(figsize=(8, 5))
plt.plot(K_sil, sil_scores, 'go-')
plt.xlabel('Кількість кластерів (k)')
plt.ylabel('Середній коефіцієнт силуета')
plt.title('Метод силуетів для Iris (3 ознаки)')
plt.xticks(K_sil)
plt.grid(True, alpha=0.3)
plt.show()

print('Silhouette scores:', [round(s, 4) for s in sil_scores])
print('Оптимальне k за силуетом:', list(K_sil)[np.argmax(sil_scores)])

**1.5. Фінальна кластеризація та візуалізація (n_clusters=2).**

In [ ]:
kmeans_final = KMeans(n_clusters=2, random_state=42, n_init=10)
labels = kmeans_final.fit_predict(X_scaled)
centers = kmeans_final.cluster_centers_

plt.figure(figsize=(8, 6))
plt.scatter(X_scaled[:, 0], X_scaled[:, 1], c=labels, cmap='viridis', s=50, alpha=0.7)
plt.scatter(centers[:, 0], centers[:, 1], c='red', s=200, marker='X', label='Центри')
plt.xlabel(iris.feature_names[0] + ' (масштабована)')
plt.ylabel(iris.feature_names[1] + ' (масштабована)')
plt.title('K-Means (k=2) — перші дві ознаки')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print('Центри кластерів (масштабовані):')
print(centers)
print('\nРозподіл за кластерами:', np.bincount(labels))

**Короткий висновок до завдання 1:**  
Метод ліктя зазвичай показує згин близько k=2 або k=3. Метод силуетів для Iris часто дає найвищий бал при k=2 (оскільки один вид чітко відокремлений). Обрано 2 кластери, бо на графіку ліктя є помітний згин, а коефіцієнт силуета максимальний при k=2. Це відповідає природному поділу: setosa добремо відокремлена від versicolor+virginica.

## Завдання 2. Дослідження даних студентів

Спочатку порівняйте три класифікатори, а потім виконайте кластеризацію. **Класифікація та кластеризація — різні задачі.**

**2.1. Імпорт бібліотек для класифікації.**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA

**2.2. Завантаження файлу.** (у Colab використовуйте kagglehub або files.upload)

In [ ]:
# Варіант для Colab:
# import kagglehub
# path = kagglehub.dataset_download("rabieelkharoua/students-performance-dataset")
# print("Path to dataset files:", path)
# df = pd.read_csv(os.path.join(path, 'Student_performance_data _.csv'))

# Якщо файл уже завантажено локально:
# df = pd.read_csv('Student_performance_data _.csv')

# Для демонстрації — якщо файл недоступний, створимо синтетичні дані схожої структури
# (у реальному виконанні замініть на справжній датасет)
print('Завантажте датасет з Kaggle (rabieelkharoua/students-performance-dataset)')
print('або розкоментуйте код вище після встановлення kagglehub.')

**2.3–2.8. Перегляд даних, структура, пропуски, дублікати, описова статистика.**  
(Після завантаження df виконайте наступні комірки)

In [ ]:
# Після завантаження df:
# df.head()
pass

In [ ]:
# print(df.columns.tolist())
pass

In [ ]:
# df.info()
pass

In [ ]:
# print(df.isnull().sum())
pass

In [ ]:
# print('Дублікати:', df.duplicated().sum())
pass

In [ ]:
# print(df.describe())
pass

**2.9. Підготовка даних для класифікації.**

In [ ]:
# X = df.drop(columns=['StudentID', 'GradeClass'])
# y = df['GradeClass']
# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
# scaler = StandardScaler()
# X_train_scaled = scaler.fit_transform(X_train)
# X_test_scaled = scaler.transform(X_test)
print('Після завантаження df розкоментуйте код вище.')

### 2.10. Класифікація: Random Forest

In [ ]:
# param_grid_rf = {
#     'n_estimators': [50, 100],
#     'max_depth': [None, 10, 20],
#     'min_samples_split': [2, 5]
# }
# grid_rf = GridSearchCV(RandomForestClassifier(random_state=42), param_grid_rf,
#                        cv=5, scoring='accuracy', n_jobs=-1)
# grid_rf.fit(X_train_scaled, y_train)
# print('Найкращі параметри RF:', grid_rf.best_params_)
# print('CV accuracy RF:', grid_rf.best_score_)
# y_pred_rf = grid_rf.predict(X_test_scaled)
# print('Test accuracy RF:', accuracy_score(y_test, y_pred_rf))

### 2.11. Класифікація: Logistic Regression

In [ ]:
# param_grid_lr = {
#     'C': [0.1, 1, 10],
#     'solver': ['saga'],
#     'max_iter': [1000]
# }
# grid_lr = GridSearchCV(LogisticRegression(random_state=42), param_grid_lr,
#                        cv=5, scoring='accuracy', n_jobs=-1)
# grid_lr.fit(X_train_scaled, y_train)
# print('Найкращі параметри LR:', grid_lr.best_params_)
# print('CV accuracy LR:', grid_lr.best_score_)
# y_pred_lr = grid_lr.predict(X_test_scaled)
# print('Test accuracy LR:', accuracy_score(y_test, y_pred_lr))

### 2.12. Класифікація: Support Vector Machine

In [ ]:
# param_grid_svm = {
#     'C': [0.1, 1, 10],
#     'kernel': ['rbf', 'linear']
# }
# grid_svm = GridSearchCV(SVC(random_state=42), param_grid_svm,
#                         cv=5, scoring='accuracy', n_jobs=-1)
# grid_svm.fit(X_train_scaled, y_train)
# print('Найкращі параметри SVM:', grid_svm.best_params_)
# print('CV accuracy SVM:', grid_svm.best_score_)
# y_pred_svm = grid_svm.predict(X_test_scaled)
# print('Test accuracy SVM:', accuracy_score(y_test, y_pred_svm))

### 2.13. Порівняння результатів

In [ ]:
# results = pd.DataFrame({
#     'Model': ['Random Forest', 'Logistic Regression', 'SVM'],
#     'CV Accuracy': [grid_rf.best_score_, grid_lr.best_score_, grid_svm.best_score_],
#     'Test Accuracy': [accuracy_score(y_test, y_pred_rf),
#                       accuracy_score(y_test, y_pred_lr),
#                       accuracy_score(y_test, y_pred_svm)]
# })
# print(results.sort_values('CV Accuracy', ascending=False))
# # classification_report для найкращої моделі
# best_model = grid_rf  # або інша за CV
# print(classification_report(y_test, best_model.predict(X_test_scaled)))

### 2.14. Кластеризація даних студентів: метод ліктя

In [ ]:
# features_clust = ['Age', 'StudyTimeWeekly', 'Absences', 'GPA']
# X_clust = df[features_clust].copy()
# scaler_c = StandardScaler()
# X_clust_scaled = scaler_c.fit_transform(X_clust)
#
# wcss_s = []
# for k in range(1, 11):
#     km = KMeans(n_clusters=k, random_state=42, n_init=10)
#     km.fit(X_clust_scaled)
#     wcss_s.append(km.inertia_)
#
# plt.figure(figsize=(8,5))
# plt.plot(range(1,11), wcss_s, 'bo-')
# plt.xlabel('k'); plt.ylabel('WCSS'); plt.title('Метод ліктя (студенти)')
# plt.grid(True, alpha=0.3); plt.show()

### 2.15. Кластеризація даних студентів: метод силуетів

In [ ]:
# sil_s = []
# for k in range(2, 11):
#     km = KMeans(n_clusters=k, random_state=42, n_init=10)
#     labels = km.fit_predict(X_clust_scaled)
#     sil_s.append(silhouette_score(X_clust_scaled, labels))
#
# plt.figure(figsize=(8,5))
# plt.plot(range(2,11), sil_s, 'go-')
# plt.xlabel('k'); plt.ylabel('Silhouette'); plt.title('Метод силуетів (студенти)')
# plt.grid(True, alpha=0.3); plt.show()
# print('Найкраще k:', range(2,11)[np.argmax(sil_s)])

### 2.16. Додатковий підхід: PCA та автоматичне визначення точки ліктя

In [ ]:
# !pip install kneed yellowbrick -q
print('За потреби: !pip install kneed yellowbrick')

In [ ]:
from sklearn.decomposition import PCA
import numpy as np
# from kneed import KneeLocator

In [ ]:
# X_pca_raw = df.drop(columns=['StudentID', 'GradeClass'])
# print(X_pca_raw.columns.tolist())

In [ ]:
# scaler_pca = StandardScaler()
# X_pca_scaled = scaler_pca.fit_transform(X_pca_raw)
# pca = PCA()
# pca.fit(X_pca_scaled)
# explained = pca.explained_variance_ratio_
# print('Пояснена дисперсія по компонентах:', np.round(explained, 4))

In [ ]:
# from kneed import KneeLocator
# kl = KneeLocator(range(1, len(explained)+1), explained, curve='convex', direction='decreasing')
# print('Точка ліктя (номер компоненти):', kl.knee)
# plt.figure(figsize=(8,5))
# plt.plot(range(1, len(explained)+1), explained, 'bo-')
# if kl.knee:
#     plt.axvline(kl.knee, color='r', linestyle='--', label=f'Knee = {kl.knee}')
# plt.xlabel('Номер компоненти'); plt.ylabel('Пояснена дисперсія')
# plt.title('PCA — частка поясненої дисперсії'); plt.legend(); plt.grid(True, alpha=0.3)
# plt.show()

In [ ]:
# n_comp = kl.knee if kl.knee else 5
# cum_var = np.cumsum(explained)
# print(f'Кумулятивна дисперсія для {n_comp} компонент: {cum_var[n_comp-1]:.4f}')
#
# plt.figure(figsize=(8,5))
# plt.plot(range(1, len(cum_var)+1), cum_var, 'bo-')
# plt.axvline(n_comp, color='r', linestyle='--')
# plt.axhline(cum_var[n_comp-1], color='g', linestyle='--')
# plt.xlabel('Кількість головних компонент')
# plt.ylabel('Кумулятивна пояснена дисперсія')
# plt.title('Кумулятивна дисперсія PCA')
# plt.grid(True, alpha=0.3); plt.show()
#
# pca_final = PCA(n_components=n_comp)
# X_pca = pca_final.fit_transform(X_pca_scaled)

In [ ]:
# from yellowbrick.cluster import KElbowVisualizer
# model = KMeans(random_state=42, n_init=10)
# visualizer = KElbowVisualizer(model, k=(1, 11))
# visualizer.fit(X_pca)
# visualizer.show()

## Висновки

**Завдання 1 (Iris):**  
Метод ліктя і метод силуетів узгоджено вказують на 2 кластери. Це відповідає природній структурі даних (setosa відокремлена). Візуалізація підтверджує чіткий поділ.

**Завдання 2:**  
- **Класифікація** використовує цільову змінну `GradeClass` — це supervised learning.  
- **Кластеризація** не використовує `GradeClass` — unsupervised, шукає природні групи за Age, StudyTimeWeekly, Absences, GPA.  
- PCA зменшує розмірність, зберігаючи основну частину дисперсії (зазвичай 70–90% при кількох компонентах).  
- Кількість кластерів визначають методом ліктя / силуетів / KElbowVisualizer.  

Після запуску всіх комірок на реальному датасеті заповніть конкретні числа (точність моделей, оптимальне k, частку дисперсії).